In [1]:
import pandas as pd
import numpy as np
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sklearn:", __import__('sklearn').__version__)
print("All imports successful.")

pandas: 3.0.2
numpy: 2.4.4
sklearn: 1.8.0
All imports successful.


In [2]:
base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"

df = pd.read_csv(os.path.join(base_dir, "data", "processed", "features.csv"))
df["timestamp"] = pd.to_datetime(df["timestamp"])

FEATURE_COLUMNS = [
    "amount",
    "km_from_last_txn",
    "minutes_from_last_txn",
    "impossible_speed",
    "hour",
    "is_weekend",
    "is_night",
    "txn_count_1hr",
    "txn_count_24hr",
    "amount_sum_1hr",
    "amount_sum_24hr",
    "amount_vs_user_avg",
    "is_new_device",
    "device_count_7d",
    "is_new_merchant_category",
]

print("Data loaded:", df.shape)
print("Feature columns:", len(FEATURE_COLUMNS))
print("Fraud transactions:", df["is_fraud"].sum())
print("Non-fraud transactions:", (df["is_fraud"] == 0).sum())

Data loaded: (78299, 25)
Feature columns: 15
Fraud transactions: 1324
Non-fraud transactions: 76975


In [3]:
X = df[FEATURE_COLUMNS].copy()
y = df["is_fraud"].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaler_path = os.path.join(base_dir, "models", "scaler.pkl")
joblib.dump(scaler, scaler_path)

print("Feature matrix shape:", X_scaled.shape)
print("Scaler saved:", scaler_path)
print("Mean of first feature before scaling:", round(X["amount"].mean(), 2))
print("Mean of first feature after scaling:", round(X_scaled[:, 0].mean(), 6))

Feature matrix shape: (78299, 15)
Scaler saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\models\scaler.pkl
Mean of first feature before scaling: 11735.66
Mean of first feature after scaling: 0.0


In [4]:
iso_forest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.025,
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

print("Training Isolation Forest on", X_scaled.shape[0], "transactions...")
iso_forest.fit(X_scaled)
print("Training complete.")

model_path = os.path.join(base_dir, "models", "isolation_forest.pkl")
joblib.dump(iso_forest, model_path)
print("Model saved:", model_path)

Training Isolation Forest on 78299 transactions...
Training complete.
Model saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\models\isolation_forest.pkl


In [5]:
df["anomaly_score_raw"] = iso_forest.decision_function(X_scaled)
df["anomaly_score"] = (df["anomaly_score_raw"] - df["anomaly_score_raw"].min()) / (df["anomaly_score_raw"].max() - df["anomaly_score_raw"].min())
df["anomaly_score"] = 1 - df["anomaly_score"]
df["predicted_fraud"] = (iso_forest.predict(X_scaled) == -1).astype(int)

print("Anomaly scores generated.")
print("Score range:", round(df["anomaly_score"].min(), 4), "to", round(df["anomaly_score"].max(), 4))
print("Transactions flagged as anomalies:", df["predicted_fraud"].sum())
print()
print("Score statistics for actual fraud transactions:")
print(df[df["is_fraud"] == 1]["anomaly_score"].describe().round(4).to_string())
print()
print("Score statistics for normal transactions:")
print(df[df["is_fraud"] == 0]["anomaly_score"].describe().round(4).to_string())

Anomaly scores generated.
Score range: 0.0 to 1.0
Transactions flagged as anomalies: 1958

Score statistics for actual fraud transactions:
count    1324.0000
mean        0.6555
std         0.1561
min         0.0880
25%         0.5426
50%         0.6729
75%         0.7783
max         1.0000

Score statistics for normal transactions:
count    76975.0000
mean         0.1721
std          0.1265
min          0.0000
25%          0.0763
50%          0.1436
75%          0.2325
max          0.8852


In [6]:
print("Classification Report:")
print(classification_report(y, df["predicted_fraud"], target_names=["Normal", "Fraud"]))

print("ROC-AUC Score:", round(roc_auc_score(y, df["anomaly_score"]), 4))

cm = confusion_matrix(y, df["predicted_fraud"])
print("Confusion Matrix:")
print("True Negatives  (Normal correctly identified):", cm[0][0])
print("False Positives (Normal flagged as fraud)    :", cm[0][1])
print("False Negatives (Fraud missed)               :", cm[1][0])
print("True Positives  (Fraud correctly caught)     :", cm[1][1])

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99     76975
       Fraud       0.46      0.67      0.54      1324

    accuracy                           0.98     78299
   macro avg       0.73      0.83      0.77     78299
weighted avg       0.99      0.98      0.98     78299

ROC-AUC Score: 0.9826
Confusion Matrix:
True Negatives  (Normal correctly identified): 75910
False Positives (Normal flagged as fraud)    : 1065
False Negatives (Fraud missed)               : 431
True Positives  (Fraud correctly caught)     : 893


In [7]:
scored_path = os.path.join(base_dir, "data", "processed", "transactions_scored.csv")
df.to_csv(scored_path, index=False)

print("Scored data saved:", scored_path)
print("Shape:", df.shape)
print("Columns added: anomaly_score_raw, anomaly_score, predicted_fraud")

Scored data saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\data\processed\transactions_scored.csv
Shape: (78299, 28)
Columns added: anomaly_score_raw, anomaly_score, predicted_fraud


In [8]:
import shap

sample_size = 2000
sample_indices = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_indices]

print("Computing SHAP values on", sample_size, "transactions...")

explainer = shap.TreeExplainer(iso_forest)
shap_values = explainer.shap_values(X_sample)

print("SHAP values computed.")
print("SHAP values shape:", shap_values.shape)

shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLUMNS)
shap_importance = shap_df.abs().mean().sort_values(ascending=False)

print()
print("Feature importance by SHAP value:")
print(shap_importance.round(6).to_string())

Computing SHAP values on 2000 transactions...
SHAP values computed.
SHAP values shape: (2000, 15)

Feature importance by SHAP value:
is_weekend                  0.502801
is_night                    0.490496
txn_count_24hr              0.304805
minutes_from_last_txn       0.259512
amount_sum_24hr             0.226156
amount_vs_user_avg          0.226032
device_count_7d             0.221005
km_from_last_txn            0.212429
amount                      0.198484
hour                        0.175814
is_new_merchant_category    0.161023
is_new_device               0.081708
impossible_speed            0.007448
amount_sum_1hr              0.005396
txn_count_1hr               0.001839


In [9]:
shap_path = os.path.join(base_dir, "models", "shap_importance.csv")
shap_df_full = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "shap_importance": shap_importance[FEATURE_COLUMNS].values
})
shap_df_full.to_csv(shap_path, index=False)

feature_columns_path = os.path.join(base_dir, "models", "feature_columns.pkl")
joblib.dump(FEATURE_COLUMNS, feature_columns_path)

print("SHAP importance saved:", shap_path)
print("Feature columns saved:", feature_columns_path)
print()
print("All model artifacts saved:")
print("  isolation_forest.pkl")
print("  scaler.pkl")
print("  shap_importance.csv")
print("  feature_columns.pkl")

SHAP importance saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\models\shap_importance.csv
Feature columns saved: C:\Users\Abhishek Shandilya\Desktop\suraksha\models\feature_columns.pkl

All model artifacts saved:
  isolation_forest.pkl
  scaler.pkl
  shap_importance.csv
  feature_columns.pkl


In [10]:
base_dir = r"C:\Users\Abhishek Shandilya\Desktop\suraksha"

df = pd.read_csv(os.path.join(base_dir, "data", "processed", "transactions_scored.csv"))
df["timestamp"] = pd.to_datetime(df["timestamp"])

iso_forest = joblib.load(os.path.join(base_dir, "models", "isolation_forest.pkl"))
scaler = joblib.load(os.path.join(base_dir, "models", "scaler.pkl"))
FEATURE_COLUMNS = joblib.load(os.path.join(base_dir, "models", "feature_columns.pkl"))
shap_importance = pd.read_csv(os.path.join(base_dir, "models", "shap_importance.csv"))

print("Scored data loaded:", df.shape)
print("Isolation Forest loaded:", iso_forest)
print("Scaler loaded:", scaler)
print("Feature columns loaded:", len(FEATURE_COLUMNS), "features")
print("SHAP importance loaded:", shap_importance.shape)

Scored data loaded: (78299, 28)
Isolation Forest loaded: IsolationForest(contamination=0.025, n_estimators=200, n_jobs=-1,
                random_state=42)
Scaler loaded: StandardScaler()
Feature columns loaded: 15 features
SHAP importance loaded: (15, 2)
